# Lesson 02 - Udforskning af Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** er en samlet ramme til at opbygge AI-agenter. Den tilbyder en ren, sammensætbar arkitektur med fire kernekomponenter:

- **Client** – opretter forbindelse til et AI-model-endpoint og håndterer kommunikation
- **Agent** – indkapsler en klient med instruktioner og værktøjsdefinitioner
- **Tools** – udvider agentens muligheder med brugerdefinerede funktioner, som modellen kan kalde
- **Session** – opretholder samtalehistorik til interaktioner med flere omgange

I denne lektion bygger vi en **rejsebookingsagent**, der tjekker destinationers tilgængelighed ved hjælp af disse koncepter.


## Opsætning


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Forståelse af Agent Framework Arkitekturen

Microsoft Agent Framework følger en lagdelt arkitektur:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – En `AzureAIProjectAgentProvider` forbinder til en Azure OpenAI-udrulning. Den håndterer autentificering, anmodningsformatering og svarparsning.
2. **Agent** – Oprettet fra klienten via `provider.create_agent()`, agenter kombinerer modeladgang med instruktioner (systemprompt) og værktøjer.
3. **Værktøjer** – Python-funktioner dekoreret med `@tool`, som agenten kan påkalde for at udføre handlinger eller hente data.
4. **Session** – Et `AgentSession` objekt (oprettet via `agent.create_session()`) der gemmer samtalehistorik og muliggør multi-turn dialog, hvor agenten husker tidligere kontekst.

Lad os bygge hvert lag trin for trin.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Tilføjelse af værktøjer med @tool-dekoratøren

Værktøjer lader agenter udføre handlinger ud over at generere tekst. `@tool`-dekoratøren konverterer en almindelig Python-funktion til noget, agenten kan kalde.

Vigtige punkter:
- Brug `Annotated[type, "description"]` så modellen forstår hver parameter.
- Docstringen bliver værktøjets beskrivelse, som modellen ser.
- `approval_mode="never_require"` betyder, at værktøjet kører automatisk uden brugerbekræftelse.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Oprettelse af en agent med værktøjer

Nu kombinerer vi klienten, instruktionerne og værktøjerne til en agent. `instructions` fungerer som systemprompten — de definerer agentens persona og adfærd.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Samtaler med flere runder ved hjælp af sessioner

En `AgentSession` (oprettet via `agent.create_session()`) holder styr på alle beskeder i en samtale. Ved at sende den samme session til hvert `agent.run()`-kald har agenten adgang til hele samtalehistorikken og kan referere tilbage til tidligere beskeder.

Vi sender `tools=[check_destination_availability]`, så agenten kan kalde vores tilgængelighedskontrol under hver runde.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Resumé

I denne lektion udforskede du de fire søjler i Microsoft Agent Framework:

| Koncept | Hvad du lærte |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` forbinder til Azure OpenAI med legitimationsbaseret godkendelse |
| **Agent** | `provider.create_agent()` pakker en modelforbindelse sammen med instruktioner og et navn |
| **Værktøjer** | `@tool` dekoratoren eksponerer Python-funktioner, som agenten kan kalde |
| **Session** | `agent.create_session()` opretholder samtalehistorik over flere runder |

Disse byggesten sammensættes til at skabe agenter, der kan føre naturlige samtaler, kalde eksterne funktioner og bevare kontekst — grundlaget for mere avancerede agentmønstre i senere lektioner.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det oprindelige dokument på dets originale sprog skal betragtes som den autoritative kilde. For vigtig information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
